# singular-matrix-mask-trick composite — cx12: swap singular LHS for a broadcast eye(3) before solve

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-repeat`, `singular-matrix-mask-trick`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "singular-matrix-mask-trick"
DD_ATOM_IDS = ["einops-repeat", "singular-matrix-mask-trick"]
DD_SUBTOPICS = ["Einops: Repeat", "Numpy: Singular matrix mask trick"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Real ARENA ray batches contain some degenerate (ray, triangle) pairs whose LHS is singular — `linalg.solve` would raise. The standard trick is to MASK them out BEFORE the solve: detect the singular rows via `|det| < eps` and overwrite their LHS with `eye(3)` (always solvable). After the solve you reset those slots to a sentinel (inf or NaN) so downstream filters skip them.

Atoms compose:
  1. `einops-repeat` broadcasts a single `eye(3)` into `(NR, 3, 3)` — stride-0 view, no copy.
  2. `singular-matrix-mask-trick`: compute the per-row det, build a singular-mask, `torch.where(mask, eye_b, LHS)` to substitute, run `linalg.solve`, then overwrite the masked rows of the result with `inf`.

The composition produces a robust solver that never throws on singular slots.

### Composite Exercise — swap singular LHS for a broadcast eye(3) before solve

**Atoms exercised together**: `einops-repeat`, `singular-matrix-mask-trick`

Implement `cx12_safe_batched_solve(LHS, RHS, eps=1e-8)` — a batched solve that gracefully handles singular matrices.

- `LHS` shape `(NR, 3, 3)`, `RHS` shape `(NR, 3)`.
- A row is 'singular' if `|det(LHS[i])| < eps`.

Steps:
1. Compute the per-row det `(NR,)` and build the mask `singular = |det| < eps`.
2. **Repeat** an `eye(3)` across the NR axis: `eye_b = repeat(t.eye(3), 'a b -> r a b', r=NR)`. Stride-0 view, no copy.
3. **Mask-substitute** LHS: where `singular[i]` is True, swap `LHS[i]` for `eye_b[i]`. Use `torch.where(singular[:, None, None], eye_b, LHS)` to broadcast the mask across the trailing dims.
4. Run `t.linalg.solve(LHS_safe, RHS)` — gives shape `(NR, 3)`.
5. Overwrite the singular slots of the result with `+inf`: `out[singular] = float('inf')`.

Return the cleaned `(NR, 3)` result.

In [ ]:
def cx12_safe_batched_solve(LHS, RHS, eps=1e-8):
    NR = LHS.shape[0]
    # Detect singular rows via |det|.
    dets = t.linalg.det(LHS)            # (NR,)
    singular = dets.abs() < eps         # (NR,) bool
    # Atom A (einops-repeat): broadcast a single eye(3) to (NR, 3, 3) — stride-0 view.
    eye_b = repeat(t.eye(3, dtype=LHS.dtype), 'a b -> r a b', r=NR)
    # Atom B (singular-matrix-mask-trick): swap singular LHS rows for eye(3) BEFORE solving.
    LHS_safe = t.where(singular[:, None, None], eye_b, LHS)
    out = t.linalg.solve(LHS_safe, RHS)
    # Sentinel the masked slots so downstream filters can drop them.
    out[singular] = float('inf')
    return out


<details><summary>Show solution — cx12</summary>

```python
def cx12_safe_batched_solve(LHS, RHS, eps=1e-8):
    NR = LHS.shape[0]
    # Detect singular rows via |det|.
    dets = t.linalg.det(LHS)            # (NR,)
    singular = dets.abs() < eps         # (NR,) bool
    # Atom A (einops-repeat): broadcast a single eye(3) to (NR, 3, 3) — stride-0 view.
    eye_b = repeat(t.eye(3, dtype=LHS.dtype), 'a b -> r a b', r=NR)
    # Atom B (singular-matrix-mask-trick): swap singular LHS rows for eye(3) BEFORE solving.
    LHS_safe = t.where(singular[:, None, None], eye_b, LHS)
    out = t.linalg.solve(LHS_safe, RHS)
    # Sentinel the masked slots so downstream filters can drop them.
    out[singular] = float('inf')
    return out
```

The eye(3)-substitution trick is the standard ARENA pattern for never-throw batched solves. Why eye(3)? Because `solve(eye, b) = b` always, regardless of `b` — so the substituted slots run through the kernel without raising, and we then overwrite them with `inf` as a sentinel. The einops repeat keeps the substitution cheap: the `(NR, 3, 3)` eye is a stride-0 view of a single 3x3 buffer, so memory cost is O(9), not O(9*NR).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx12'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx12',
        'subtopics': ["Einops: Repeat", "Numpy: Singular matrix mask trick"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()